In [8]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI



In [9]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

openai = OpenAI(api_key=openrouter_api_key, base_url='https://openrouter.ai/api/v1')
ollama  = OpenAI(api_key=openrouter_api_key, base_url="http://localhost:11434/v1")
OPENROUTER_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "llama3.2"


In [10]:
system_prompt = """
You are a helpful assistant that can answer technical questions and help with tasks pertaining to the topic of the user's question and it should be in the language of the user's question.
"""


In [11]:
def stream_llama(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [12]:
def stream_openai(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [13]:
def stream_model(prompt, model):
    if model=="LLAMA":
        result = stream_llama(prompt)
    elif model=="OpenAI":
        result = stream_openai(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [16]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["LLAMA", "OpenAI"], label="Select model", value="LLAMA")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
     title="Technical Q&A Assistant",
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
     examples=[
        ["What is a Python decorator? Give a short example.", "LLAMA"],
        ["Explain recursion in one paragraph.", "OpenAI"]
    ],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
